# Entrepinares · Cuaderno 2: el workflow con Foundry

Aquí el workflow **se ejecuta de verdad** con un modelo de Microsoft Foundry. Verás, para cada pregunta: qué ruta eligió el Orquestador, qué agentes intervinieron, qué herramientas llamaron y la respuesta final.

**⚠️ Antes de empezar (esto SÍ usa red y consume recursos de Azure)**
- Cada pregunta hace **2-4 llamadas** al modelo (`gpt-4.1-mini`): coste muy bajo, pero real.
- Necesitas: haber hecho **`az login`** en tu propio ordenador con tu cuenta autorizada y un archivo **`.env`** con `FOUNDRY_PROJECT_ENDPOINT` (y, si procede, `AZURE_AI_MODEL_DEPLOYMENT_NAME` y `AZURE_TENANT_ID`). Ver `docs/GUIA_INSTALACION_Y_EJECUCION.md`.
- ⚠️ **Iniciar sesión no concede permisos**: tu usuario necesita un rol en el proyecto Foundry que permita usar el modelo. Si falta, el cuaderno lo dirá con un error 401/403; **no se sustituye por ninguna simulación**.
- 🔒 No escribas credenciales en las celdas: la autenticación la gestiona Azure CLI y nunca se imprime ningún token.
- 🟢 El workflow es de **solo lectura**: no registra herramientas de escritura y **no modifica datos**. Al final se verifica con SHA-256.
- Las celdas de los apartados 1-3 **no llaman al modelo**; las llamadas empiezan en el apartado 4.

**Importante:** el modelo de lenguaje no es determinista. Una misma pregunta puede, de vez en cuando, tomar otra ruta (lo veremos en el apartado 6). Es una característica de los LLM, no un fallo del cuaderno.

## 1. Preparación y diagnóstico de configuración

Carga el `.env` (solo si existe y sin pisar variables ya definidas), bloquea las descargas de CISA y guarda la huella de los datos. **No hace ninguna llamada de red.**

In [ ]:
import sys
from pathlib import Path

# Localiza la raíz del repositorio (funciona abriendo el cuaderno desde notebooks/ o desde la raíz)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos" / "inventario_demo.csv").exists())
sys.path.insert(0, str(RAIZ / "notebooks"))
import entrepinares_utils as u

huellas_antes = u.huellas_datos()
m = u.preparar_entorno()          # carga .env (si existe) y bloquea descargas CISA

cfg = u.comprobar_configuracion()
print("Archivo .env                    :", "✔ existe" if cfg["archivo_env_existe"] else "✘ no existe (copia .env.example a .env)")
print("FOUNDRY_PROJECT_ENDPOINT        :", ("✔ " + cfg["host_endpoint"]) if cfg["endpoint_configurado"] else "✘ NO definido")
print("AZURE_AI_MODEL_DEPLOYMENT_NAME  :", cfg["modelo"])
print("AZURE_TENANT_ID (opcional)      :", "definido" if cfg["tenant_indicado"] else "no definido (se usa el tenant de az login)")
print("Azure CLI (az) en el PATH       :", "✔" if cfg["az_en_path"] else "✘ instálala y reabre VS Code/JupyterLab")

## 2. Comprobar la sesión de Azure CLI *(opcional)*

Pide **un token** para Foundry con tu sesión de `az login`. No usa el modelo ni consume cuota y **nunca se imprime el token**. Que funcione demuestra que tienes sesión, **no** que tengas permisos en el proyecto (eso se ve en el apartado 4).

In [ ]:
try:
    print(u.probar_credencial())
except Exception as e:
    print("❌", u.explicar_error(e))

## 3. Construir el workflow (solo lectura)

`construir_workflow(acciones_interactivas=False)` crea los tres agentes y el grafo. Construirlo **no llama al modelo**; las llamadas ocurren al ejecutar. Comprobamos qué herramientas tiene cada agente: no aparece ninguna de escritura.

In [ ]:
try:
    workflow, agentes = m.construir_workflow(acciones_interactivas=False)
except Exception as e:
    print("❌", u.explicar_error(e))
    raise

for nombre, agente in agentes.items():
    print(f"{nombre}: {[t.name for t in agente.default_options.get('tools', [])]}")
u.mostrar_mermaid(u.diagrama_mermaid(workflow))

## 4. Ejecutar consultas y leer la traza

Definimos `preguntar(texto)`: construye un workflow **nuevo** por pregunta (para que no se arrastre estado entre casos), lo ejecuta con `await` y muestra la **traza**:

- **RUTA**: los nodos que se ejecutaron, en orden (eventos `executor_invoked` de MAF).
- **DECISIÓN DEL ORQUESTADOR**: el objeto tipado del que depende el enrutamiento.
- **HERRAMIENTAS LLAMADAS**: las llamadas a funciones que aparecen en los mensajes de cada agente. Si el SDK no las expone en algún caso, se indica `(ninguna visible en los eventos)`; no significa que no se llamaran.
- **RESPUESTA FINAL**: lo que ve el usuario.

In [ ]:
trazas = []

async def preguntar(texto):
    try:
        t = await u.ejecutar_consulta(texto, modulo=m)
    except Exception as e:
        print("❌ ERROR:", u.explicar_error(e))
        raise
    trazas.append(t)
    u.mostrar_traza(t)

### Caso A · Pregunta sencilla (esperado: respuesta directa, sin agentes especialistas)

In [ ]:
await preguntar("¿Qué versión tiene IT-004?")

### Caso B · Correlación (esperado: Orquestador → Correlación)
Pide candidatas de CISA KEV, sin pedir análisis ni prioridad.

In [ ]:
await preguntar("¿Qué CVE candidatas aparecen para IT-004 en CISA KEV?")

### Caso C · Análisis avanzado (esperado: Orquestador → Correlación → Criticidad)
Aquí intervienen los **tres agentes**.

In [ ]:
await preguntar("Analiza las vulnerabilidades y la prioridad de IT-004.")

### Caso D · Activo inexistente (esperado: no se llama a Correlación)

In [ ]:
await preguntar("¿Qué versión tiene IT-999?")

### Caso E · Una CVE concreta sobre un activo con hotfix
Esperado: un análisis completo donde la CVE aparece como **corregida** según nuestras reglas (inventario ficticio), recordando que eso no implica ausencia de otras vulnerabilidades.

> **Observa la traza:** si el Orquestador indica la CVE, el cribado se filtra **solo por esa CVE**. Eso no equivale a analizar todas las candidatas del activo. Comprueba si la respuesta generaliza más de lo que devuelven las herramientas.

In [ ]:
await preguntar("¿Está afectado IT-006 por CVE-2024-3400? ¿Qué otros riesgos podrían existir?")

### Caso F · Pedir una acción de escritura (en solo lectura no se ejecuta)
El Orquestador reconoce la petición, pero este workflow **no registra** la herramienta de actualización: la respuesta final debe explicar que **no se ha ejecutado nada**. (La actualización real con aprobación humana existe en la consola del script original y no forma parte de esta formación.)

> El mensaje final menciona DevUI porque es el mismo texto para DevUI y para este cuaderno: los dos funcionan en solo lectura.

In [ ]:
await preguntar("Actualiza ahora las novedades de CISA KEV.")

## 5. Comparativa de las ejecuciones

In [ ]:
u.tabla_resumen(trazas)

## 6. Variabilidad del enrutamiento

Los LLM no son deterministas. Para una pregunta **ambigua** el Orquestador puede elegir rutas distintas en ejecuciones distintas. Repetimos una pregunta de frontera y miramos solo la **ruta elegida y los agentes**. No corregimos esa variabilidad: queremos verla.

> Consumo: `REPETICIONES` × (2-4 llamadas al modelo). Con 3 repeticiones sigue siendo muy bajo.

In [ ]:
REPETICIONES = 3
PREGUNTA_FRONTERA = "Dame una visión de los riesgos del firewall IT-004."

repetidas = []
for i in range(REPETICIONES):
    t = await u.ejecutar_consulta(PREGUNTA_FRONTERA, modulo=m)
    repetidas.append(t)
    print(f"Ejecución {i + 1}: ruta={t.decision['ruta'] if t.decision else '?'} | agentes={' -> '.join(t.agentes)} | {t.duracion_s}s")

rutas = {(t.decision or {}).get("ruta") for t in repetidas}
print("\nRutas distintas observadas:", rutas)
print("Conclusión:", "hubo variabilidad" if len(rutas) > 1 else "esta vez la ruta fue estable (puede variar en otra sesión)")

## 7. Integridad de los datos

Si el workflow de solo lectura hizo lo que prometía, ni un solo archivo de `datos/` ha cambiado.

In [ ]:
ok = u.informe_integridad(huellas_antes, u.huellas_datos())
assert ok, "¡Algún dato ha cambiado! Restaura el repositorio con git."

## Resumen

- Cada pregunta produjo una **traza**: ruta, decisión tipada, herramientas y respuesta.
- Las preguntas sencillas se resuelven con **un agente**; las de análisis recorren **tres**.
- El enrutamiento depende del LLM y **puede variar**.
- Todo fue **solo lectura**, comprobado por SHA-256.
- El **mismo workflow** está desplegado como agente **Hosted V5** en Foundry (versión publicada v6): para consultarlo de forma remota usa `scripts/probar_hosted_v5.py` (pasos 9-12 de la guía y `docs/HOSTED_V5.md`).

---
# BONUS · Ver el workflow en DevUI (puerto 8091)

DevUI es una interfaz web de desarrollo para ver el grafo y los eventos en vivo. **No se ejecuta dentro de este cuaderno**: bloquea el proceso. Ábrelo en **otra terminal de PowerShell**, en la raíz del repositorio, con el venv activado (`.\.venv\Scripts\Activate.ps1`) y tras `az login`.

**⚠️ Seguridad:** DevUI escucha **solo en `127.0.0.1`** (tu propio equipo). No lo expongas a la red.

**Opción A — con autenticación (por defecto, recomendada):** DevUI exige un *token Bearer*. Define uno propio en esa terminal antes de arrancar:
```powershell
$env:DEVUI_AUTH_TOKEN = python -c "import secrets; print(secrets.token_urlsafe(32))"
$env:DEVUI_AUTH_TOKEN
python scripts\lanzar_devui.py --puerto 8091
```
Abre <http://127.0.0.1:8091>. Si la interfaz pide un token, pega el valor que mostró la segunda línea (no lo compartas ni lo guardes en archivos).

**Opción B — sin autenticación** (solo en tu equipo, para una demo):
```powershell
python scripts\lanzar_devui.py --puerto 8091 --sin-auth
```

Para parar DevUI, pulsa `Ctrl + C` en esa terminal. DevUI usa tu sesión de `az login` y consume llamadas al modelo cuando le haces preguntas. En DevUI el workflow es igualmente de solo lectura.